In [26]:
import glob

In [30]:
len(glob.glob('drive_videos_fixed/*'))

100

In [37]:
for i in glob.glob('drive_videos_fixed/*'):
    print(i.split('/')[-1].split('__')[0].lower())

evan_williams
sea_glass
cupcake_orange
aa_batt
columbia_valley
duct_tape
clif_peanut_butter_banana
happy_day
gruet
gaucho_blue
clif_chocolate
twisted_tea_black_cherry
finest_call_huckleberry
cinnamon_toast_crunch
little_tree_royal_pine
makers_mark
tie_downs
gaucho_white
zingers_raspberry
jack_daniel_s_honey
finest_call_sweet
tire_gauge
bulbs
hand_warmers
gaucho_yellow
cupcake_chocolate
korbel_brut
clif_crunchy
buckhead_green
blackwoods_russian_cream_5
twisted_tea_extreme
nature_valley_peanut
clif_white_chocolate
black_velvet
segrams_7
little_tree_black_ice
tire_repair_kit
twinkies
chateau_michelle_riesling
clif_cool_mint
jack_daniels
finest_call_mango
mikes_harder_strawberry
super_glue
barefoot_riesling
mikes_harder_lemonade
twisted_tea
twisted_tea_half_and_half
nature_valley
segrams_vo
chips_deluxe
twisted_tea_peach
jack_daniel_s_apple
canadian_mist
mikes_harder_black_cherry
zingers_vanilla
little_tree_new_car
r_and_r
electrical_tape
mikes_harder_mango
hogue
hakutsuru
twisted_tea_extr

In [40]:
import cv2
import os

def extract_60_frames_from_videos(input_folder, output_folder, target_frames=60):
    """
    Extract exactly `target_frames` high-quality frames across all videos in a folder.
    Evenly sample from the total frames of all videos combined.
    """
    os.makedirs(output_folder, exist_ok=True)

    folder_name = os.path.basename(os.path.normpath(input_folder))
    folder_prefix = folder_name.replace(" ", "_")

    video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.flv')
    videos = [os.path.join(input_folder, f) for f in sorted(os.listdir(input_folder))
              if f.lower().endswith(video_extensions)]

    if not videos:
        print(f"⚠️ No videos found in '{input_folder}'")
        return

    # Count total frames across all videos
    total_frames_all = 0
    frame_counts = []
    for video in videos:
        cap = cv2.VideoCapture(video)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        frame_counts.append(total)
        total_frames_all += total
        cap.release()

    if total_frames_all == 0:
        print(f"❌ No readable frames in '{input_folder}'")
        return

    # Determine which global frame indices to capture
    step = max(1, total_frames_all // target_frames)
    selected_indices = set(i * step for i in range(target_frames))
    print(f"\n🎬 Processing folder: {folder_name}")
    print(f"   Total frames: {total_frames_all}, Step: {step}, Frames to save: {len(selected_indices)}")

    frame_count_global = 1
    global_frame_index = 0
    saved = 0

    for video_path in videos:
        cap = cv2.VideoCapture(video_path)
        video_name = os.path.basename(video_path)
        total_in_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

        print(f"   ▶ Reading {video_name} ({total_in_video} frames)")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if global_frame_index in selected_indices:
                # ---- Save High Quality ----
                filename_out = f"{folder_prefix}_frame_{frame_count_global:03d}.png"
                out_path = os.path.join(output_folder, filename_out)

                # PNG is lossless; set compression low for speed and high fidelity
                cv2.imwrite(out_path, frame, [cv2.IMWRITE_PNG_COMPRESSION, 0])

                # If you prefer JPG (smaller size but still good quality):
                # cv2.imwrite(out_path.replace('.png', '.jpg'), frame, [cv2.IMWRITE_JPEG_QUALITY, 100])

                saved += 1
                frame_count_global += 1

                if saved >= target_frames:
                    print(f"✅ Reached {target_frames} frames — stopping early.")
                    cap.release()
                    print(f"🎉 Done! Saved {saved} high-quality frames from folder '{folder_name}'")
                    return

            global_frame_index += 1

        cap.release()

    print(f"🎉 Done! Saved {saved} high-quality frames from folder '{folder_name}'")


def process_all_video_folders(base_input_dir, base_output_dir, target_frames=60):
    """
    Loop through all folders in base_input_dir.
    Extract exactly `target_frames` frames per folder total.
    Skip folders that already exist.
    """
    os.makedirs(base_output_dir, exist_ok=True)

    for folder in sorted(os.listdir(base_input_dir)):
        input_path = os.path.join(base_input_dir, folder)
        if not os.path.isdir(input_path):
            continue  # skip non-folder items

        safe_folder_name = folder.replace(" ", "_")
        output_path = os.path.join(base_output_dir, safe_folder_name)

        if os.path.exists(output_path):
            print(f"⏩ Skipping '{folder}' (output folder '{safe_folder_name}' already exists)")
            continue

        extract_60_frames_from_videos(input_path, output_path, target_frames)


# === Your Directories ===
base_input_dir = "/home/ubuntu/additional_drive/shwan_data/scripts/drive_videos_fixed/"
base_output_dir = "/home/ubuntu/additional_drive/shwan_data/scripts/all_frames/"
target_frames = 60  # capture 60 frames total per folder

process_all_video_folders(base_input_dir, base_output_dir, target_frames)


⏩ Skipping '7_days_chocolate__48794101015__8' (output folder '7_days_chocolate__48794101015__8' already exists)
⏩ Skipping '7_days_mini_croissants__816374020670__5' (output folder '7_days_mini_croissants__816374020670__5' already exists)
⏩ Skipping '7_days_strawberry__816374020281__2' (output folder '7_days_strawberry__816374020281__2' already exists)
⏩ Skipping '7_days_vanilla__48794101022__4' (output folder '7_days_vanilla__48794101022__4' already exists)
⏩ Skipping 'AAA_batt__73096500273__12' (output folder 'AAA_batt__73096500273__12' already exists)
⏩ Skipping 'AA_Batt__73096500235__13' (output folder 'AA_Batt__73096500235__13' already exists)
⏩ Skipping 'Blackwoods_Russian_cream_5__71610302792__4' (output folder 'Blackwoods_Russian_cream_5__71610302792__4' already exists)
⏩ Skipping 'Blackwoods_Russian_cream__71610303065__24' (output folder 'Blackwoods_Russian_cream__71610303065__24' already exists)

🎬 Processing folder: Blackwoods_honey__71610301832__10
   Total frames: 1631, Ste


🎬 Processing folder: jack_Daniel_s_apple__82184004371__9
   Total frames: 2290, Step: 38, Frames to save: 60
   ▶ Reading 20251029_133229.mp4 (932 frames)
   ▶ Reading 20251029_133303.mp4 (349 frames)
   ▶ Reading 20251029_133311.mp4 (259 frames)
   ▶ Reading 20251029_135532.mp4 (542 frames)
   ▶ Reading 20251029_135544.mp4 (208 frames)
✅ Reached 60 frames — stopping early.
🎉 Done! Saved 60 high-quality frames from folder 'jack_Daniel_s_apple__82184004371__9'

🎬 Processing folder: jack_Daniel_s_honey__82184000335__10
   Total frames: 2361, Step: 39, Frames to save: 60
   ▶ Reading 20251029_133353.mp4 (961 frames)
   ▶ Reading 20251029_133412.mp4 (375 frames)
   ▶ Reading 20251029_133421.mp4 (418 frames)
   ▶ Reading 20251029_135550.mp4 (353 frames)
   ▶ Reading 20251029_135559.mp4 (254 frames)
✅ Reached 60 frames — stopping early.
🎉 Done! Saved 60 high-quality frames from folder 'jack_Daniel_s_honey__82184000335__10'

🎬 Processing folder: jack_Daniels__82184090466__11
   Total frames:

In [41]:
import os

def remove_empty_folders(base_dir):
    """
    Recursively remove all empty folders inside base_dir.
    """
    removed_count = 0

    # Walk from bottom up so subfolders are deleted before parents
    for root, dirs, files in os.walk(base_dir, topdown=False):
        for d in dirs:
            dir_path = os.path.join(root, d)
            try:
                if not os.listdir(dir_path):  # folder is empty
                    os.rmdir(dir_path)
                    print(f"🗑️ Removed empty folder: {dir_path}")
                    removed_count += 1
            except Exception as e:
                print(f"⚠️ Could not remove {dir_path}: {e}")

    print(f"\n✅ Done! Total empty folders removed: {removed_count}")


# === Example Usage ===
base_dir = "/home/ubuntu/additional_drive/shwan_data/scripts/frames_captue_new/"
remove_empty_folders(base_dir)



✅ Done! Total empty folders removed: 0


In [42]:
pwd

'/home/ubuntu/additional_drive/shwan_data/scripts'